# Vibration Signal Analysis
## Machine Health Monitoring & Fault Diagnosis System

**Notebook 03** — deep-dive into vibration signal processing:

1. Synthetic signal generation (healthy vs fault)
2. Time-domain waveform comparison
3. FFT spectrum analysis
4. Bearing fault frequency calculation (BPFI, BPFO, BSF, FTF)
5. Envelope analysis — Hilbert demodulation
6. Spectrogram (Short-Time Fourier Transform)
7. Feature extraction across fault types
8. Feature separability (box plots, PCA projection)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.fft import rfft, rfftfreq
from scipy.signal import hilbert, butter, filtfilt, spectrogram
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

FS = 25600   # Hz — typical vibration acquisition rate
T  = 2       # seconds per signal
N  = FS * T  # total samples
t  = np.linspace(0, T, N, endpoint=False)

print(f'Signal parameters: Fs={FS} Hz, Duration={T}s, N={N:,} samples')

## 1. Signal Generation

In [ ]:
np.random.seed(42)

# Bearing geometry (SKF 6205-2RS typical values)
shaft_rpm   = 1500
shaft_hz    = shaft_rpm / 60     # 25 Hz
n_balls     = 9
ball_dia_mm = 7.94
pitch_dia_mm= 38.50
contact_angle = 0.0              # degrees (deep groove)

# Bearing fault frequencies
ratio = (ball_dia_mm / pitch_dia_mm) * np.cos(np.radians(contact_angle))
BPFO  = shaft_hz * (n_balls / 2) * (1 - ratio)   # Ball Pass Freq, Outer race
BPFI  = shaft_hz * (n_balls / 2) * (1 + ratio)   # Ball Pass Freq, Inner race
BSF   = shaft_hz * (pitch_dia_mm / (2 * ball_dia_mm)) * (1 - ratio**2)  # Ball Spin
FTF   = shaft_hz * (1 - ratio) / 2               # Fundamental Train Freq

print('Bearing Fault Frequencies (Hz):')
print(f'  Shaft (1X)  : {shaft_hz:.2f} Hz')
print(f'  BPFO        : {BPFO:.2f} Hz  ({BPFO/shaft_hz:.2f}X)')
print(f'  BPFI        : {BPFI:.2f} Hz  ({BPFI/shaft_hz:.2f}X)')
print(f'  BSF         : {BSF:.2f} Hz  ({BSF/shaft_hz:.2f}X)')
print(f'  FTF         : {FTF:.2f} Hz  ({FTF/shaft_hz:.2f}X)')

In [ ]:
def make_signal(fault_type='healthy', snr_db=20):
    """Generate a synthetic vibration signal for a given fault type."""
    # Base: shaft harmonics
    sig = (np.sin(2*np.pi*shaft_hz*t) +
           0.3*np.sin(2*np.pi*shaft_hz*2*t) +
           0.15*np.sin(2*np.pi*shaft_hz*3*t))

    if fault_type == 'outer_race':          # BPFO impulsing
        impulse_times = np.arange(0, T, 1/BPFO)
        for ti in impulse_times:
            idx = int(ti * FS)
            if idx < N - 50:
                sig[idx:idx+50] += 4.0 * np.exp(-np.linspace(0, 10, 50))

    elif fault_type == 'inner_race':        # BPFI impulsing (modulated by shaft)
        impulse_times = np.arange(0, T, 1/BPFI)
        modulation    = np.sin(2*np.pi*shaft_hz*impulse_times) + 1
        for ti, mod in zip(impulse_times, modulation):
            idx = int(ti * FS)
            if idx < N - 50:
                sig[idx:idx+50] += 3.5 * mod * np.exp(-np.linspace(0, 10, 50))

    elif fault_type == 'ball':              # BSF impulsing
        impulse_times = np.arange(0, T, 1/BSF)
        for ti in impulse_times:
            idx = int(ti * FS)
            if idx < N - 50:
                sig[idx:idx+50] += 2.5 * np.exp(-np.linspace(0, 10, 50))

    elif fault_type == 'unbalance':         # 1X dominant
        sig += 3.0 * np.sin(2*np.pi*shaft_hz*t + 0.5)

    elif fault_type == 'misalignment':      # 1X + 2X dominant
        sig += 2.5 * np.sin(2*np.pi*shaft_hz*2*t + 1.0)

    # Add noise at specified SNR
    signal_power = np.mean(sig**2)
    noise_power  = signal_power / (10**(snr_db/10))
    noise = np.sqrt(noise_power) * np.random.randn(N)
    return sig + noise

signals = {
    'Healthy':     make_signal('healthy',    snr_db=25),
    'Outer Race':  make_signal('outer_race', snr_db=20),
    'Inner Race':  make_signal('inner_race', snr_db=20),
    'Ball Fault':  make_signal('ball',       snr_db=20),
    'Unbalance':   make_signal('unbalance',  snr_db=22),
    'Misalignment':make_signal('misalignment',snr_db=22),
}
print(f'Generated {len(signals)} signals, each {N:,} samples @ {FS} Hz')

## 2. Time-Domain Waveform Comparison

In [ ]:
colors = ['#27ae60','#e74c3c','#e67e22','#8e44ad','#2980b9','#f39c12']
WINDOW = int(0.1 * FS)   # show first 100 ms

fig, axes = plt.subplots(len(signals), 1, figsize=(14, 10), sharex=True)

for ax, (name, sig), color in zip(axes, signals.items(), colors):
    ax.plot(t[:WINDOW]*1000, sig[:WINDOW], color=color, linewidth=0.7)
    ax.set_ylabel(name, fontsize=9, rotation=0, ha='right', va='center', labelpad=70)
    rms_val = np.sqrt(np.mean(sig**2))
    ax.set_title(f'RMS={rms_val:.3f}  Kurt={stats.kurtosis(sig, fisher=False):.2f}',
                 fontsize=8, loc='right', pad=2, color='gray')
    ax.set_yticks([])
    ax.spines[['top','right','left']].set_visible(False)

axes[-1].set_xlabel('Time (ms)')
fig.suptitle('Vibration Waveforms — First 100 ms (all signals)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. FFT Spectrum Comparison

In [ ]:
def compute_spectrum(sig, fs=FS):
    yf = np.abs(rfft(sig)) * 2 / len(sig)
    xf = rfftfreq(len(sig), d=1/fs)
    return xf, yf

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (name, sig), color in zip(axes, signals.items(), colors):
    xf, yf = compute_spectrum(sig)

    ax.plot(xf, yf, color=color, linewidth=0.6, alpha=0.9)

    # Mark shaft harmonics
    for h in range(1, 5):
        ax.axvline(shaft_hz*h, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)

    # Mark fault frequency if applicable
    freq_markers = {'Outer Race': BPFO, 'Inner Race': BPFI, 'Ball Fault': BSF}
    if name in freq_markers:
        f_fault = freq_markers[name]
        ax.axvline(f_fault, color='red', linestyle='--', linewidth=1,
                   label=f'Fault freq: {f_fault:.1f} Hz')
        ax.legend(fontsize=8)

    ax.set_xlim(0, 600)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude')

fig.suptitle('FFT Spectra — Fault Type Comparison (0–600 Hz)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Envelope Analysis (Hilbert Demodulation)

In [ ]:
def envelope_analysis(sig, fs=FS, bp_low=2000, bp_high=8000):
    """Band-pass → Hilbert → envelope spectrum."""
    b, a = butter(4, [bp_low/(fs/2), bp_high/(fs/2)], btype='bandpass')
    sig_bp = filtfilt(b, a, sig)
    env    = np.abs(hilbert(sig_bp))
    xf, yf = compute_spectrum(env, fs)
    return env, xf, yf

fig, axes = plt.subplots(3, 2, figsize=(14, 9))
fault_signals = {k: v for k, v in signals.items() if k != 'Healthy'}
healthy_sig   = signals['Healthy']

for ax_pair, (name, sig) in zip(axes, fault_signals.items()):
    env_h, xf_h, yf_h = envelope_analysis(healthy_sig)
    env_f, xf_f, yf_f = envelope_analysis(sig)

    ax = ax_pair[0]
    ax.plot(xf_h[:int(len(xf_h)*0.02)], yf_h[:int(len(yf_h)*0.02)],
            color='#27ae60', linewidth=0.8, label='Healthy', alpha=0.7)
    ax.plot(xf_f[:int(len(xf_f)*0.02)], yf_f[:int(len(yf_f)*0.02)],
            color='#e74c3c', linewidth=0.8, label=name)
    ax.set_title(f'Envelope Spectrum — {name}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude')
    ax.legend(fontsize=8)

    freq_markers = {'Outer Race': BPFO, 'Inner Race': BPFI, 'Ball Fault': BSF}
    if name in freq_markers:
        ax.axvline(freq_markers[name], color='purple', linestyle='--',
                   linewidth=1, label=f'{freq_markers[name]:.1f} Hz')

    # Time envelope
    ax2 = ax_pair[1]
    wn  = int(0.05 * FS)
    ax2.plot(t[:wn]*1000, sig[:wn],    color='#2980b9', linewidth=0.6, alpha=0.5, label='Signal')
    ax2.plot(t[:wn]*1000, env_f[:wn],  color='#e74c3c', linewidth=1.5, label='Envelope')
    ax2.set_title(f'Signal + Envelope — {name}', fontsize=9, fontweight='bold')
    ax2.set_xlabel('Time (ms)')
    ax2.legend(fontsize=8)

plt.suptitle('Envelope Analysis — Bearing Fault Detection', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Spectrogram (STFT)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for ax, (name, sig), color in zip(axes, signals.items(), colors):
    f_spec, t_spec, Sxx = spectrogram(sig, fs=FS, nperseg=512, noverlap=256)
    # Show up to 1000 Hz
    freq_mask = f_spec <= 1000
    im = ax.pcolormesh(t_spec, f_spec[freq_mask],
                       10*np.log10(Sxx[freq_mask] + 1e-12),
                       cmap='inferno', shading='gouraud')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    plt.colorbar(im, ax=ax, label='dB')

fig.suptitle('Spectrograms (0–1000 Hz) — STFT Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Feature Extraction Across All Fault Types

In [ ]:
from src.features.feature_extraction import extract_all_features

rows = []
for name, sig in signals.items():
    feats = extract_all_features(sig, fs=FS, label=name)
    rows.append(feats)

feat_df = pd.DataFrame(rows).set_index('label')
print('Extracted features per fault type:')
print(feat_df.round(4).to_string())

In [ ]:
# Heatmap of normalised feature values
from sklearn.preprocessing import MinMaxScaler

feat_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(feat_df),
    index=feat_df.index,
    columns=feat_df.columns
)

import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(feat_norm.T, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalised value'})
ax.set_title('Feature Values by Fault Type (normalised 0–1)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fault Type')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## 7. PCA Projection (Feature Separability)

In [ ]:
# Generate many windows per fault type for a richer PCA
from src.preprocessing.clean_data import segment_signal

all_feats, all_labels = [], []
for name, sig in signals.items():
    segs = segment_signal(sig, window_size=512, overlap=0.5)
    for seg in segs:
        all_feats.append(extract_all_features(seg, fs=FS))
        all_labels.append(name)

feat_all_df = pd.DataFrame(all_feats).drop(columns=['label'], errors='ignore')
labels_arr  = np.array(all_labels)

# Standardise and project to 2D
X_scaled = StandardScaler().fit_transform(feat_all_df.fillna(0))
pca      = PCA(n_components=2, random_state=42)
X_pca    = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1','PC2'])
pca_df['Fault'] = labels_arr

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}  PC2={pca.explained_variance_ratio_[1]:.1%}')

fig, ax = plt.subplots(figsize=(10, 7))
for (fault, group), color in zip(pca_df.groupby('Fault'), colors):
    ax.scatter(group['PC1'], group['PC2'], c=color, label=fault,
               s=18, alpha=0.6, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Projection of Vibration Features — Fault Separability',
             fontsize=13, fontweight='bold')
ax.legend(markerscale=2, framealpha=0.9)
plt.tight_layout()
plt.show()

## Summary

| Fault | Dominant Indicator | Key Frequency |
|---|---|---|
| Healthy | Low kurtosis (~3), low crest factor | Shaft 1X only |
| Outer Race | High kurtosis, regular impulses at BPFO | BPFO = `{BPFO:.1f}` Hz |
| Inner Race | High kurtosis, AM-modulated impulses at BPFI | BPFI = `{BPFI:.1f}` Hz |
| Ball Fault | Moderate kurtosis, BSF tone in envelope | BSF = `{BSF:.1f}` Hz |
| Unbalance | High RMS, dominant 1X | 1X = 25 Hz |
| Misalignment | High RMS, dominant 2X/3X | 2X = 50 Hz |